# Chapter 10: Coverage Modifications

Code of Chapter 10 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 10.7 Python Walkthrough

In [ ]:
import numpy as np
from gidata import synthetic_portfolio
from gilossmodels import fit_mle
from gicoverage import (lev, lev_empirical, survival, payment, per_loss, per_payment,
                        ler, franchise_per_loss, inflate, layer_cost)

# Section 10.2.4: Pareto policy, d = 500, u = 25,000, c = 0.8
P = {"alpha": 3, "theta": 8000}
lev("pareto", P, 500), lev("pareto", P, 25_000)       # 456.75, 3764.92
per_loss("pareto", P, 500, 25_000, 0.8)              # 2646.54
per_payment("pareto", P, 500, 25_000, 0.8)           # 3174.43

# Section 10.2.5: empirical and lognormal limited expected values
policies, claims = synthetic_portfolio()
x = claims["ultimate_amount"].to_numpy()
L = fit_mle(x, "lognormal")
[round(lev_empirical(x, d), 2) for d in (250, 500, 1000, 2500)]
[round(ler("lognormal", L, d), 4) for d in (250, 500, 1000, 2500)]

# Section 10.3: inflation
per_loss("pareto", inflate("pareto", P, 0.10), 500, 25_000, 0.8)     # 2913.08
[round(layer_cost("lognormal", inflate("lognormal", L, r), 10_000, 10_000), 2)
 for r in (0, 0.05, 0.10)]                                          # 481.32, 523.62, 566.65

# Section 10.4: per-policy cost and payment frequency
0.097 * per_loss("lognormal", L, 500, 25_000)        # 319.00
0.097 * survival("lognormal", L, 500)                 # 0.0856
payment(x, 500, 25_000).mean()                         # empirical per-loss payment

## Exercise 1: The memoryless deductible

In [ ]:
from gicoverage import ler, per_payment, per_loss, inflate
E = {"beta": 1 / 2000}
ler("exponential", E, 250), per_payment("exponential", E, 250)        # 0.1175, 2000.0
per_loss("exponential", inflate("exponential", E, 0.10), 250)         # 1963.68

## Exercise 3: Pricing an unindexed layer

In [ ]:
from gidata import synthetic_portfolio
from gilossmodels import fit_mle
from gicoverage import layer_cost, inflate
_, claims = synthetic_portfolio()                  # the 2025 claims of Section 10.2.5
x = claims["ultimate_amount"].to_numpy()
L = fit_mle(x, "lognormal")
485 * layer_cost("lognormal", L, 10_000, 10_000)                          # 233442
485 * layer_cost("lognormal", inflate("lognormal", L, 1.05**2 - 1), 10_000, 10_000)   # 275875